# 02 Training — RT-DETR-l

This notebook trains and resumes an RT-DETR-l object detector on the manually annotated 4-class VOC subset. It is designed for Google Colab with Google Drive persistence, so checkpoints and outputs survive session resets.

In [13]:
!nvidia-smi

Tue Mar 10 06:23:22 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [14]:
# Cell 2 — install

!pip install -q ultralytics pyyaml

In [15]:
# Cell 3 — imports and Drive mount

from pathlib import Path
import os
import shutil
import zipfile
import json
import time

import yaml
import pandas as pd

from google.colab import drive
drive.mount('/content/drive')

from ultralytics import RTDETR

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [16]:
# Cell 4 — config (RT-DETR-specific)

# ===== USER CONFIG =====
DRIVE_ROOT = Path("/content/drive/MyDrive/object_detection_rtdetr")
BUNDLE_ZIP_PATH = DRIVE_ROOT / "training_bundle.zip"
WORKSPACE_DIR = Path("/content/workspace_rtdetr")
EXTRACT_DIR = WORKSPACE_DIR / "project_bundle"

MODEL_NAME = "rtdetr-l.pt"
MODEL_KIND = "rtdetr"
RUN_PROJECT = str(DRIVE_ROOT / "runs")
RUN_NAME = "rtdetr_l_voc4"

SEED = 42
IMG_SIZE = 640
EPOCHS_TARGET = 60          # first target
BATCH_SIZE = 8             # safer default for RT-DETR-l on Colab GPU
PATIENCE = 20
OPTIMIZER = "AdamW"        # sensible performance-first choice for transformer-style training
LR0 = 0.0001               # aligns with assignment default scale
DEVICE = 0

FORCE_REEXTRACT_BUNDLE = False

In [17]:
# Cell 5 — helpers

def bytes_to_gb(num_bytes: int) -> float:
    return num_bytes / (1024 ** 3)


def print_disk_usage(path: Path) -> None:
    usage = shutil.disk_usage(path)
    print(f"Disk usage for {path}:")
    print(f"  total: {bytes_to_gb(usage.total):.2f} GB")
    print(f"  used : {bytes_to_gb(usage.used):.2f} GB")
    print(f"  free : {bytes_to_gb(usage.free):.2f} GB")


def extract_bundle_if_needed(bundle_zip_path: Path, extract_dir: Path, force: bool = False) -> None:
    if not bundle_zip_path.exists():
        raise FileNotFoundError(f"Bundle zip not found: {bundle_zip_path}")

    expected_yaml = extract_dir / "data" / "data.yaml"

    if expected_yaml.exists() and not force:
        print(f"Bundle already extracted: {extract_dir}")
        return

    if force and extract_dir.exists():
        shutil.rmtree(extract_dir)

    extract_dir.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(bundle_zip_path, "r") as zf:
        zf.extractall(extract_dir)

    if not expected_yaml.exists():
        raise FileNotFoundError(f"Extraction finished but missing data.yaml at: {expected_yaml}")

    print(f"Bundle extracted to: {extract_dir}")


def read_data_yaml(data_yaml_path: Path) -> dict:
    with open(data_yaml_path, "r", encoding="utf-8") as f:
        return yaml.safe_load(f)


def find_existing_run_dir(run_project: str, run_name: str) -> Path:
    run_dir = Path(run_project) / run_name
    return run_dir


def get_checkpoint_paths(run_dir: Path) -> dict:
    weights_dir = run_dir / "weights"
    return {
        "run_dir": run_dir,
        "weights_dir": weights_dir,
        "last": weights_dir / "last.pt",
        "best": weights_dir / "best.pt",
    }


def checkpoint_exists(path: Path) -> bool:
    return path.exists() and path.is_file() and path.stat().st_size > 0

In [18]:
# Cell 6 — workspace setup and space check

DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
WORKSPACE_DIR.mkdir(parents=True, exist_ok=True)

print_disk_usage(Path("/content"))
print_disk_usage(DRIVE_ROOT)

if not BUNDLE_ZIP_PATH.exists():
    raise FileNotFoundError(
        f"Upload training_bundle.zip to Google Drive first.\nMissing: {BUNDLE_ZIP_PATH}"
    )

print(f"Bundle zip size: {bytes_to_gb(BUNDLE_ZIP_PATH.stat().st_size):.2f} GB")

Disk usage for /content:
  total: 112.64 GB
  used : 43.63 GB
  free : 68.99 GB
Disk usage for /content/drive/MyDrive/object_detection_rtdetr:
  total: 15.00 GB
  used : 0.03 GB
  free : 14.97 GB
Bundle zip size: 0.03 GB


In [19]:
# Cell 7 — extract bundle

extract_bundle_if_needed(
    bundle_zip_path=BUNDLE_ZIP_PATH,
    extract_dir=EXTRACT_DIR,
    force=FORCE_REEXTRACT_BUNDLE,
)

DATA_YAML_PATH = EXTRACT_DIR / "data" / "data.yaml"
data_yaml = read_data_yaml(DATA_YAML_PATH)

print("Loaded data.yaml:")
print(json.dumps(data_yaml, indent=2))

Bundle already extracted: /content/workspace_rtdetr/project_bundle
Loaded data.yaml:
{
  "path": "F:\\Projects\\Object_Detection_Computer_Vision\\data",
  "train": "images/train",
  "val": "images/val",
  "test": "images/test",
  "names": {
    "0": "car",
    "1": "bus",
    "2": "bicycle",
    "3": "motorbike"
  }
}


In [20]:
# Cell 8 — patch data.yaml for Colab runtime

PATCHED_DATA_YAML_PATH = EXTRACT_DIR / "data" / "data_colab.yaml"

patched_data_yaml = {
    "path": str((EXTRACT_DIR / "data").resolve()),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "names": data_yaml["names"],
}

with open(PATCHED_DATA_YAML_PATH, "w", encoding="utf-8") as f:
    yaml.safe_dump(patched_data_yaml, f, sort_keys=False, allow_unicode=True)

print(f"Patched Colab data.yaml written to: {PATCHED_DATA_YAML_PATH}")
print(json.dumps(patched_data_yaml, indent=2))

Patched Colab data.yaml written to: /content/workspace_rtdetr/project_bundle/data/data_colab.yaml
{
  "path": "/content/workspace_rtdetr/project_bundle/data",
  "train": "images/train",
  "val": "images/val",
  "test": "images/test",
  "names": {
    "0": "car",
    "1": "bus",
    "2": "bicycle",
    "3": "motorbike"
  }
}


In [21]:
# Cell 9 — checkpoint discovery

run_dir = find_existing_run_dir(RUN_PROJECT, RUN_NAME)
ckpt = get_checkpoint_paths(run_dir)

print("Run directory:", ckpt["run_dir"])
print("Last checkpoint exists:", checkpoint_exists(ckpt["last"]))
print("Best checkpoint exists:", checkpoint_exists(ckpt["best"]))

Run directory: /content/drive/MyDrive/object_detection_rtdetr/runs/rtdetr_l_voc4
Last checkpoint exists: False
Best checkpoint exists: False


In [22]:
# Cell 10 — pre-training evaluation helper

def validate_model(model_path_or_name, data_yaml_path, split="test"):
    model = RTDETR(str(model_path_or_name))
    metrics = model.val(
        data=str(data_yaml_path),
        split=split,
        imgsz=IMG_SIZE,
        batch=BATCH_SIZE,
        device=DEVICE,
        plots=False,
    )
    return metrics


def metrics_to_dict(metrics, label: str) -> dict:
    return {
        "label": label,
        "map50": float(metrics.box.map50),
        "map50_95": float(metrics.box.map),
        "mp": float(metrics.box.mp),
        "mr": float(metrics.box.mr),
    }

In [23]:
# Cell 11 — evaluate existing checkpoint before training if present

pretrain_metrics = None

if checkpoint_exists(ckpt["best"]):
    print("Evaluating existing best checkpoint before training...")
    metrics_before = validate_model(
        model_path_or_name=ckpt["best"],
        data_yaml_path=PATCHED_DATA_YAML_PATH,
        split="test",
    )
    pretrain_metrics = metrics_to_dict(metrics_before, label="before_training")
    print(pretrain_metrics)
else:
    print("No existing best checkpoint found. Fresh training run will start from pretrained weights.")

No existing best checkpoint found. Fresh training run will start from pretrained weights.


In [24]:
# Cell 12 — training cell (RT-DETR)

if checkpoint_exists(ckpt["last"]):
    print(f"Resuming training from: {ckpt['last']}")
    model = RTDETR(str(ckpt["last"]))
    train_results = model.train(
        data=str(PATCHED_DATA_YAML_PATH),
        epochs=EPOCHS_TARGET,
        imgsz=IMG_SIZE,
        batch=BATCH_SIZE,
        seed=SEED,
        device=DEVICE,
        project=RUN_PROJECT,
        name=RUN_NAME,
        optimizer=OPTIMIZER,
        lr0=LR0,
        patience=PATIENCE,
        resume=True,
        save=True,
        save_period=-1,
        plots=True,
    )
else:
    print(f"Starting fresh training from pretrained weights: {MODEL_NAME}")
    model = RTDETR(MODEL_NAME)
    train_results = model.train(
        data=str(PATCHED_DATA_YAML_PATH),
        epochs=EPOCHS_TARGET,
        imgsz=IMG_SIZE,
        batch=BATCH_SIZE,
        seed=SEED,
        device=DEVICE,
        project=RUN_PROJECT,
        name=RUN_NAME,
        optimizer=OPTIMIZER,
        lr0=LR0,
        patience=PATIENCE,
        save=True,
        save_period=-1,
        plots=True,
    )

print("Training complete.")

Starting fresh training from pretrained weights: rtdetr-l.pt
Ultralytics 8.4.21 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/workspace_rtdetr/project_bundle/data/data_colab.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=60, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=rtdetr-l.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=rtdetr_l_voc4, nbs=64,

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


       1/60      6.62G      1.622      3.637      2.657          3        640: 100% ━━━━━━━━━━━━ 14/14 1.5s/it 20.4s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 3.5s/it 6.9s
                   all         22        117   0.000265     0.0761    0.00176   0.000284

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
       2/60      7.02G      2.247     0.1835      4.043         82        640: 0% ──────────── 0/14  0.6s

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


       2/60      7.02G      1.636     0.6419      1.916          3        640: 100% ━━━━━━━━━━━━ 14/14 1.7it/s 8.2s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 3.9it/s 0.5s
                   all         22        117   7.72e-05     0.0455   0.000186   5.41e-05

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
       3/60      7.02G      1.012      1.256     0.6558         66        640: 0% ──────────── 0/14  0.8s

...

      58/60      7.02G     0.5073      1.546     0.2943          3        640: 100% ━━━━━━━━━━━━ 14/14 1.4it/s 10.1s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 2.6it/s 0.8s
                   all         22        117      0.523       0.42      0.158      0.081

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
      59/60      7.02G     0.5048      1.396      0.216         51        640: 0% ──────────── 0/14  0.7s

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      59/60      7.02G     0.4577      1.585     0.2454          2        640: 100% ━━━━━━━━━━━━ 14/14 1.3it/s 11.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 4.5it/s 0.4s
                   all         22        117      0.523       0.42      0.149     0.0785

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
      60/60      7.02G     0.4956      1.378     0.2721         54        640: 0% ──────────── 0/14  0.7s

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      60/60      7.02G     0.4814      1.519     0.2637         14        640: 100% ━━━━━━━━━━━━ 14/14 1.7it/s 8.3s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 3.9it/s 0.5s
                   all         22        117      0.524      0.442      0.173     0.0852

60 epochs completed in 0.220 hours.
Optimizer stripped from /content/drive/MyDrive/object_detection_rtdetr/runs/rtdetr_l_voc4/weights/last.pt, 66.2MB
Optimizer stripped from /content/drive/MyDrive/object_detection_rtdetr/runs/rtdetr_l_voc4/weights/best.pt, 66.2MB

Validating /content/drive/MyDrive/object_detection_rtdetr/runs/rtdetr_l_voc4/weights/best.pt...
Ultralytics 8.4.21 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
rt-detr-l summary: 310 layers, 31,991,960 parameters, 0 gradients, 103.4 GFLOPs
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 3.8it/s 0.5s
                   a